# Мой запуск публичного extra и собственные упражнения
Исходный публичный блокнот прочитан. Ниже — его реальный повтор, затем мои упражнения. Потоки1. Данные в ../project/data. Ожидаю: ранняя остановка выберет разные числа деревьев при разных lr; прогнозы не обязаны улучшиться. Перед запуском финальных значений своих ещё не знаю.


In [1]:
import os
THREADS = 1                                     # все библиотеки на одном потоке, см. пояснение выше
os.environ["OMP_NUM_THREADS"] = str(THREADS)    # задаём до импорта sklearn и lightgbm

import time
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import lightgbm as lgb
import catboost
from catboost import CatBoostRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

RANDOM_STATE = 0
DATA = next(p for p in [Path("../../data"), Path("../data"), Path("../project/data"), Path(".")]
            if (p / "daily_sales.csv").exists())

print("pandas", pd.__version__, "| sklearn", sklearn.__version__,
      "| lightgbm", lgb.__version__, "| catboost", catboost.__version__)
print("данные:", DATA.resolve())

pandas 3.0.6 | sklearn 1.8.0 | lightgbm 4.7.0 | catboost 1.2.10
данные: /Users/dmitriikormin/Documents/Projects/ds-course/docs/reaudit-2026-11-16/ch10/student/project/data


## 1. Данные и признаки

Признаки те же, что в функции `prep` из проекта: температура, дождь, праздник, акция, день недели, месяц, выходной, номер дня `day_n`, кофейня и формат. Разница одна: кофейню и формат оставляем **текстом**, без `get_dummies`. Каждая библиотека получит категории в том виде, который понимает сама.

Для апрельского подбора и июльского выпуска строим два прогноза погоды из истории, доступной на каждую дату. После даты выпуска температуру и долю дождливых дней заменяем историческими средними для календарного месяца.

In [2]:
s = pd.read_csv(DATA / "daily_sales.csv", parse_dates=["date"])
s = s.sort_values(["shop_id", "date"]).reset_index(drop=True)

def prep(s, issued):
    s = s.copy()
    history = s[s["date"] < issued]
    future = s["date"] >= issued
    for field in ("temp", "rain"):
        s[field] = s[field].astype(float)
        monthly = history.groupby(history["date"].dt.month)[field].mean()
        s.loc[future, field] = s.loc[future, "date"].dt.month.map(monthly)
    s["weekday"] = s["date"].dt.weekday
    s["month"] = s["date"].dt.month
    s["weekend"] = (s["weekday"] >= 5).astype(int)
    s["day_n"] = (s["date"] - pd.Timestamp("2024-01-01")).dt.days
    cols = ["temp", "rain", "holiday", "promo", "weekday", "month", "weekend", "day_n", "shop_id", "format"]
    return s[cols]

CAT = ["shop_id", "format"]
X_dev = prep(s, "2025-04-01")
X = prep(s, "2025-07-01")                                                     # категории — строки (для CatBoost)
X_cat = X.astype({c: "category" for c in CAT})                  # тип category (для LightGBM и HGB)
dev_cat = X_dev.astype({c: X_cat[c].dtype for c in CAT})
dev_dum = pd.get_dummies(X_dev, columns=CAT, dtype=int)
X_dum = pd.get_dummies(X, columns=CAT, dtype=int)               # флаги (классический вариант)
y = s["revenue_k"]

print("строк:", len(X), "| признаков:", X.shape[1], "| столбцов после get_dummies:", X_dum.shape[1])
X.head()

строк: 4386 | признаков: 10 | столбцов после get_dummies: 17


,temp,rain,holiday,promo,weekday,month,weekend,day_n,shop_id,format
0,-19.0,0.0,1,0,0,1,0,0,KZN-1,улица
1,-14.4,1.0,1,0,1,1,0,1,KZN-1,улица
2,-10.8,0.0,1,0,2,1,0,2,KZN-1,улица
3,-16.8,1.0,1,0,3,1,0,3,KZN-1,улица
4,-18.7,0.0,1,0,4,1,0,4,KZN-1,улица


**Вывод по разделу 1.** Один набор признаков в трёх видах: строки, тип `category` и флаги. После `get_dummies` столбцов становится 17 вместо 10. На шести кофейнях это не страшно, но с тысячами товаров или адресов таблица флагов раздулась бы до тысяч почти пустых столбцов.

## 2. Разбиение по времени

Как в проекте: прогноз нужен на будущее, поэтому проверяем «прошлое → будущее».

- **обучение** (`fit`) — до 1 апреля 2025;
- **валидация** (`val`) — апрель–июнь 2025: по ней ранняя остановка выбирает число деревьев;
- **тест** (`test`) — июль–декабрь 2025: его модели не видят до финальной оценки.

Тест для ранней остановки брать нельзя. Иначе число деревьев подобрано по тесту, и оценка на нём получится оптимистичной.

In [3]:
fit = s["date"] < "2025-04-01"
val = (s["date"] >= "2025-04-01") & (s["date"] < "2025-07-01")
test = s["date"] >= "2025-07-01"

for name, mask in [("fit", fit), ("val", val), ("test", test)]:
    print(f"{name:5} {mask.sum():5} строк, {s.loc[mask, 'date'].min():%Y-%m-%d} — {s.loc[mask, 'date'].max():%Y-%m-%d}")
refit = fit | val

fit    2736 строк, 2024-01-01 — 2025-03-31
val     546 строк, 2025-04-01 — 2025-06-30
test   1104 строк, 2025-07-01 — 2025-12-31


**Вывод по разделу 2.** Три непересекающихся периода идут друг за другом. Все модели учатся на одном и том же `fit`, останавливаются по одному `val` и проверяются на одном `test`, поэтому сравнение честное.

## 3. Обучение и замер

Для каждой модели запишем строку таблицы: качество на тесте (RMSE и MAE, тыс. ₽), смещение (средняя ошибка «факт − прогноз»: больше нуля — модель занижает), сколько деревьев выбрала ранняя остановка, время обучения и время прогноза. Секундомер — `time.perf_counter()`, как в главе 9. Прогноз быстрый, поэтому его повторяем 5 раз и берём лучшее время.

In [4]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def timed(f, repeat=1):
    # Запускает f() repeat раз; возвращает результат и лучшее время в секундах.
    best = None
    for _ in range(repeat):
        t0 = time.perf_counter()
        result = f()
        t = time.perf_counter() - t0
        best = t if best is None else min(best, t)
    return result, best

rows = []

def add_row(name, model, X_test, n_trees, fit_s, search_s):
    pred, pred_s = timed(lambda: model.predict(X_test), repeat=5)
    rows.append({"модель": name, "RMSE": rmse(y[test], pred), "MAE": mean_absolute_error(y[test], pred),
                 "смещение": float((y[test] - pred).mean()),        # > 0: прогноз в среднем занижен
                 "деревьев": int(n_trees), "final fit, с": fit_s, "подбор и final fit, с": search_s, "predict, мс": pred_s * 1000})
    print(rows[-1])

### HistGradientBoosting из sklearn

Встроенная ранняя остановка HGB берёт валидацию случайной долей строк, а нам нужна валидация по времени. Поэтому делаем, как в шаге 6 проекта: обучаем 1000 деревьев, по `staged_predict` находим лучшее число деревьев на `val` и переобучаем модель с этим числом. Таблица показывает отдельно время подбора с переобучением и время последнего обучения на всех данных до июля.

In [5]:
def hgb_by_val(dev, final, **extra):
    started = time.perf_counter()
    params = dict(learning_rate=0.05, max_iter=1000, early_stopping=False, random_state=RANDOM_STATE, **extra)
    probe = HistGradientBoostingRegressor(**params).fit(dev[fit], y[fit])
    curve = [rmse(y[val], p) for p in probe.staged_predict(dev[val])]
    params["max_iter"] = int(np.argmin(curve)) + 1
    model, fit_s = timed(lambda: HistGradientBoostingRegressor(**params).fit(final[refit], y[refit]))
    return model, params["max_iter"], fit_s, time.perf_counter()-started

m, k, t, total_s = hgb_by_val(dev_dum, X_dum)
add_row("HGB, get_dummies", m, X_dum[test], k, t, total_s)
m, k, t, total_s = hgb_by_val(dev_cat, X_cat, categorical_features="from_dtype")
add_row("HGB, категории", m, X_cat[test], k, t, total_s)

{'модель': 'HGB, get_dummies', 'RMSE': 9.13104580022069, 'MAE': 6.763525121287867, 'смещение': -0.006904993569007939, 'деревьев': 67, 'final fit, с': 0.05403641698649153, 'подбор и final fit, с': 0.7297626660438254, 'predict, мс': 1.3347080093808472}


{'модель': 'HGB, категории', 'RMSE': 9.12125191232744, 'MAE': 6.676548081477782, 'смещение': 0.71910042770172, 'деревьев': 56, 'final fit, с': 0.035966041032224894, 'подбор и final fit, с': 0.5975264169974253, 'predict, мс': 2.053416974376887}


### LightGBM

Категории LightGBM понимает по типу `category`. Валидацию передают в `eval_X` и `eval_y` (так с версии 4.7; в старых версиях — `eval_set=[(X_val, y_val)]`). Ранняя остановка — обратный вызов `lgb.early_stopping(100)`: если 100 деревьев подряд не улучшили ошибку на валидации, обучение останавливается. Лучшее число деревьев хранится в `best_iteration_`, и `predict` использует именно его.

Обучим две версии. Первая — с настройками по умолчанию: до 31 листа в дереве и не меньше 20 строк в листе. Эти настройки рассчитаны на большие таблицы. Вторая — с мелкими деревьями: 7 листьев и 10 строк в листе.

In [6]:
for name, leaves, min_rows in [("LightGBM", 31, 20), ("LightGBM, 7 листьев", 7, 10)]:
    started = time.perf_counter()
    lg = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.05, num_leaves=leaves, min_child_samples=min_rows,
                           random_state=RANDOM_STATE, verbose=-1, n_jobs=THREADS)
    _, fit_s = timed(lambda: lg.fit(dev_cat[fit], y[fit], eval_X=dev_cat[val], eval_y=y[val],
                                    callbacks=[lgb.early_stopping(100, verbose=False)]))
    chosen = lg.best_iteration_
    lg = lgb.LGBMRegressor(n_estimators=chosen, learning_rate=.05, num_leaves=leaves, min_child_samples=min_rows, random_state=RANDOM_STATE, verbose=-1,n_jobs=THREADS)
    _, final_fit = timed(lambda: lg.fit(X_cat[refit],y[refit]))
    add_row(name, lg, X_cat[test], chosen, final_fit, time.perf_counter()-started)

{'модель': 'LightGBM', 'RMSE': 12.645920884334704, 'MAE': 8.775164178214181, 'смещение': 5.836016552608142, 'деревьев': 120, 'final fit, с': 0.01988662499934435, 'подбор и final fit, с': 0.05884475004859269, 'predict, мс': 2.128165971953422}
{'модель': 'LightGBM, 7 листьев', 'RMSE': 13.261974504340218, 'MAE': 9.264577363782799, 'смещение': 6.256568122506405, 'деревьев': 330, 'final fit, с': 0.01878054201370105, 'подбор и final fit, с': 0.04339858400635421, 'predict, мс': 3.101125010289252}


### CatBoost

CatBoost получает категории строками и список их имён в `cat_features`. Внутри он заменяет категорию средним ответом по ней, посчитанным только по «предыдущим» строкам: так ответ текущей строки не подсказывает сам себе. `early_stopping_rounds=100` и `use_best_model=True` — та же ранняя остановка по `eval_set`. `allow_writing_files=False` — не создавать рядом с блокнотом папку `catboost_info` с логами обучения.

In [7]:
started = time.perf_counter()
cb = CatBoostRegressor(iterations=3000, learning_rate=0.05, depth=6, random_seed=RANDOM_STATE,
                       verbose=0, early_stopping_rounds=100, thread_count=THREADS, allow_writing_files=False)
_, fit_s = timed(lambda: cb.fit(X_dev[fit], y[fit], cat_features=CAT,
                                eval_set=(X_dev[val], y[val]), use_best_model=True))
chosen = cb.get_best_iteration()+1
cb = CatBoostRegressor(iterations=chosen,learning_rate=.05,depth=6,random_seed=RANDOM_STATE,verbose=0,thread_count=THREADS,allow_writing_files=False)
_, final_fit = timed(lambda: cb.fit(X[refit],y[refit],cat_features=CAT))
add_row("CatBoost",cb,X[test],chosen,final_fit,time.perf_counter()-started)

{'модель': 'CatBoost', 'RMSE': 9.01712455294474, 'MAE': 6.45837867815739, 'смещение': 0.23598996631807453, 'деревьев': 225, 'final fit, с': 0.23189433303195983, 'подбор и final fit, с': 0.6032931670197286, 'predict, мс': 0.6498750299215317}


**Вывод по разделу 3.** Все пять моделей обучены на одних данных и остановлены по одной валидации. Число деревьев у них разное: у каждой библиотеки свой размер и своё устройство дерева. Поэтому сравнивать модели надо по качеству на тесте и по времени, а не по числу деревьев.

## 4. Сравнение

In [8]:
table = pd.DataFrame(rows).set_index("модель").sort_values("RMSE")
table.round({"RMSE": 2, "MAE": 2, "смещение": 2, "fit, с": 2, "predict, мс": 1})

,RMSE,MAE,смещение,деревьев,"final fit, с","подбор и final fit, с","predict, мс"
модель,,,,,,,
CatBoost,9.02,6.46,0.24,225,0.231894,0.603293,0.6
"HGB, категории",9.12,6.68,0.72,56,0.035966,0.597526,2.1
"HGB, get_dummies",9.13,6.76,-0.01,67,0.054036,0.729763,1.3
LightGBM,12.65,8.78,5.84,120,0.019887,0.058845,2.1
"LightGBM, 7 листьев",13.26,9.26,6.26,330,0.018781,0.043399,3.1


In [9]:
imp = pd.Series(cb.get_feature_importance(), index=X.columns).sort_values(ascending=False)
imp.round(1)

weekday    26.7
format     19.0
shop_id    17.4
weekend    10.9
rain        8.0
holiday     6.8
promo       6.0
day_n       2.5
temp        1.8
month       0.9
dtype: float64

**Прочитайте свою таблицу.** Сравните ошибки финального полугодия, число деревьев и смещение. Объясните, где бюджет ограничивает обучение, а где — число пакетных прогнозов. Время подбора включает раннюю остановку и финальное переобучение. Колонка final fit показывает только второй шаг. Итоги периода понадобятся для отчёта о качестве уже выбранных настроек.

## 5. Попробуйте сами

1. Поменяйте `learning_rate` на 0.1 и 0.02 у LightGBM и CatBoost. Как меняются число деревьев, время и RMSE?
2. Передайте в CatBoost таблицу `X_dum` без `cat_features`. Хуже ли стало качество?
3. Перенесите сюда приём из шага 7 проекта: обучите бустинги на `y / trend` без `day_n` и сравните RMSE.
4. Попробуйте у LightGBM `num_leaves=15` и `min_child_samples=5`. У LightGBM дерево растёт «по листьям», и его сложность задаёт `num_leaves`, а не глубина.

## Мои упражнения по разделу5
Q1: проверить влияние lr, кодирования категорий, тренда и листьев. Q2: доступны параметры fit, val, итоговый test. Q3: число деревьев изменится; тренд должен снизить занижение; ни один параметр заранее не объявляю победителем. Выбор упражнения фиксирован до его test-вывода.


Перед повтором: в первом собственном дизайне seed11/patience60/cap1200 отличались от публичного контроля. Сохранил первый notebook+CSV отдельно. Для чистого сравнения меняю только контрольные настройки на общиеseed0/patience100/cap3000;8заранее выбранных вопросов сохраняются. Ожидаю увидеть эффекты параметров при равных остальных условиях, без гарантии улучшения test.


In [10]:
exercise_rows=[]
from sklearn.linear_model import LinearRegression

def exercise(kind,lr=.05,leaves=31,min_rows=20,dummies=False,detrend=False):
    dev=dev_dum.copy() if dummies else (X_dev.copy() if kind=='CatBoost' else dev_cat.copy())
    final=X_dum.copy() if dummies else (X.copy() if kind=='CatBoost' else X_cat.copy())
    target_dev=y.copy(); target_final=y.copy(); multiplier=np.ones(len(y))
    if detrend:
        td=LinearRegression().fit(dev.loc[fit,['day_n']],np.log(y[fit]))
        tf=LinearRegression().fit(final.loc[refit,['day_n']],np.log(y[refit]))
        target_dev=y/np.exp(td.predict(dev[['day_n']]))
        target_final=y/np.exp(tf.predict(final[['day_n']]))
        multiplier=np.exp(tf.predict(final[['day_n']]))
        dev=dev.drop(columns='day_n'); final=final.drop(columns='day_n')
    started=time.perf_counter()
    if kind=='CatBoost':
        params=dict(iterations=3000,learning_rate=lr,depth=6,random_seed=0,verbose=0,thread_count=1,allow_writing_files=False)
        m=CatBoostRegressor(**params)
        cats=[] if dummies else CAT
        m.fit(dev[fit],target_dev[fit],cat_features=cats,eval_set=(dev[val],target_dev[val]),early_stopping_rounds=100,use_best_model=True)
        chosen=m.get_best_iteration()+1
        params['iterations']=chosen; m=CatBoostRegressor(**params)
        start=time.perf_counter(); m.fit(final[refit],target_final[refit],cat_features=cats); fit_s=time.perf_counter()-start
    else:
        params=dict(n_estimators=3000,learning_rate=lr,num_leaves=leaves,min_child_samples=min_rows,random_state=0,verbose=-1,n_jobs=1)
        m=lgb.LGBMRegressor(**params)
        m.fit(dev[fit],target_dev[fit],eval_X=dev[val],eval_y=target_dev[val],callbacks=[lgb.early_stopping(100,verbose=False)])
        chosen=m.best_iteration_; params['n_estimators']=chosen; m=lgb.LGBMRegressor(**params)
        start=time.perf_counter(); m.fit(final[refit],target_final[refit]); fit_s=time.perf_counter()-start
    selection_and_final_s=time.perf_counter()-started
    start=time.perf_counter(); p=m.predict(final[test])*multiplier[test]; predict_ms=(time.perf_counter()-start)*1000
    exercise_rows.append(dict(model=kind,lr=lr,leaves=leaves,dummies=dummies,detrend=detrend,trees=chosen,rmse_k=rmse(y[test],p),mae_k=mean_absolute_error(y[test],p),bias_k=float(np.mean(y[test]-p)),selection_and_final_s=selection_and_final_s,final_fit_s=fit_s,predict_ms=predict_ms))

for kind in ['LightGBM','CatBoost']:
    for rate in [.1,.02]: exercise(kind,lr=rate)
exercise('CatBoost',dummies=True)
exercise('CatBoost',detrend=True)
exercise('LightGBM',detrend=True)
exercise('LightGBM',leaves=15,min_rows=5)
exercise_table=pd.DataFrame(exercise_rows)
exercise_table.to_csv('exercises.csv',index=False)
table.to_csv('public-repeat.csv')
print(exercise_table.to_string(index=False))


   model   lr  leaves  dummies  detrend  trees    rmse_k    mae_k    bias_k  selection_and_final_s  final_fit_s  predict_ms
LightGBM 0.10      31    False    False    595 13.203626 9.410599  6.447576               0.197121     0.087960   13.663375
LightGBM 0.02      31    False    False    325 12.757887 8.906288  5.797991               0.110330     0.048038    6.594083
CatBoost 0.10      31    False    False     96  9.128105 6.666029  0.466889               0.249740     0.067726    0.864791
CatBoost 0.02      31    False    False    752  9.089471 6.461683  0.151957               1.509918     0.741596    1.245167
CatBoost 0.05      31     True    False    164  8.944401 6.418363  0.312172               0.238909     0.094554    0.815625
CatBoost 0.05      31    False     True    144  8.920886 6.422583 -1.042609               0.282422     0.087504    0.987000
LightGBM 0.05      31    False     True     73 12.050973 8.776392  2.344834               0.034918     0.011420    1.671042
LightGBM

## Мой вывод после реального запуска
Публичный повтор: CatBoost RMSE9.017тыс.₽/225деревьев, HGBкатегории9.121/56, HGBфлаги9.131/67; LightGBM12.646/120 и7листов13.262/330. Версии:sklearn1.8.0,CatBoost1.2.10,LightGBM4.7.0,потоки1.
При общихseed0/patience100/cap3000 CatBoost lr.1 выбрал96деревьев сRMSE9.128, lr.02 —752сRMSE9.089: меньший шаг дороже и не обеспечил улучшение относительно.05. LightGBM lr.1 выбрал595деревьев/RMSE13.204, lr.02 —325/RMSE12.758: правило о двукратном числе деревьев лишь ориентир; конкретная кривая и остановка могут вести себя иначе.
CatBoost с флагами далRMSE8.944 против9.017категорий в этом полугодии — флаги здесь не ухудшили результат. После снятия тренда RMSE CatBoost8.921 иLightGBM12.051; уCatBoost смещение стало−1.043 вместо+0.236, поэтому меньший RMSE не означает меньший модуль среднего смещения. УLightGBM смещение уменьшилось с+5.836 до+2.345.
LightGBM15листов/min_child_samples5 далRMSE12.555 против12.646контроля; улучшение в одном полугодии не доказательство превосходства настройки вообще. Все8упражнений заранее заданы публичными вопросами, после них свою проектную модель не перевыбираю. Финальный test здесь служит учебному сравнению: для нового производственного выбора понадобится следующий нетронутый период.
Реальные selection+final и final fit, пакетный predict лежат вCSV. УCatBoost.02 финальный fit0.742с против.068с при.1; уLightGBM.1 пакетный predict13.66мс против6.59мс при.02 — число деревьев сказывается на выдаче. Восемь вариантов измерены одним запуском/компьютером; это не стабильный рейтинг скоростей на любой машине.
